# Evaluation RAG avec RAGAS

Evaluation du systeme de recherche de tickets via le framework RAGAS.

**Metriques :**
- **LLMContextRecall** : les documents retrouves contiennent-ils l'information attendue ?
- **Faithfulness** : la reponse est-elle fidele aux documents retrouves ?

*(On demarre avec 2 metriques pour stabiliser le pipeline, puis on ajoutera LLMContextPrecisionWithoutReference et ResponseRelevancy.)*

**LLM Judge** : `mistralai/mistral-nemo` via OpenRouter  
**Embeddings** : `sentence-transformers/all-MiniLM-L6-v2` (local, pas d'appel OpenAI)

## 1. Setup

In [4]:
import os
import sys
from dotenv import load_dotenv

load_dotenv()

sys.path.append(os.path.abspath("../app"))
from rag_engine import search

## 2. Configuration LLM evaluateur + Embeddings

In [1]:
import os
from openai import OpenAI
from ragas.llms import llm_factory
from ragas.embeddings.base import embedding_factory
from ragas.metrics.collections import ContextRecall, Faithfulness
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper


OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
DENSE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY manquante")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "ragas-eval",
    },
)

evaluator_llm = llm_factory("mistralai/mistral-nemo", client=client)

dense_embeddings = HuggingFaceEmbeddings(
    model_name=DENSE_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)
evaluator_embeddings = LangchainEmbeddingsWrapper(dense_embeddings)

metrics = [
    ContextRecall(llm=evaluator_llm),
    Faithfulness(llm=evaluator_llm),
]

c:\Users\mahmo\OneDrive\Documents\RAG-LogiStore\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\mahmo\OneDrive\Documents\RAG-LogiStore\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mahmo\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run 

## 3. Jeu de requetes test

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/tickets_clean.csv")
df_en = df[(df["language"] == "en") & (df["subject"].notna())].sample(n=3, random_state=123)

test_cases = [
    {"query": row["subject"], "reference": row["answer"]}
    for _, row in df_en.iterrows()
]

import json
print(json.dumps(test_cases, indent=2, ensure_ascii=False))

[
  {
    "query": "Inquiry About Secure Medical Data Solutions",
    "reference": "I would be happy to provide information regarding secure medical data solutions for hospital services, including billing integration. These services can help simplify the billing process while maintaining the security of patient data. I will send a detailed brochure, and if you prefer, we can schedule a call to discuss further. Please let me know a convenient time."
  },
  {
    "query": "Enhancing Multi-Unit Marketing Processes",
    "reference": "Thank you for your inquiry, [client]. To provide relevant assistance, could you please specify which analytics and automation tools your teams are currently using? This will enable us to suggest compatible solutions, effective practices, and relevant case studies tailored to your environment."
  },
  {
    "query": "Enhance Investment Analysis Software",
    "reference": "Thank you for your suggestion to enhance investment analysis tools. To better address yo

## 4. Collecte des resultats RAG

In [9]:
def collect_results(test_cases, method="hybrid", limit=3):
    dataset = []
    for tc in test_cases:
        results = search(tc["query"], method=method, limit=limit)
        contexts = [r["body"] for r in results] if results else []
        response = results[0]["answer"] if results else "" 

        dataset.append({
            "user_input": tc["query"],           
            "retrieved_contexts": contexts,     
            "response": response,               
            "reference": tc["reference"],       
        })
    return dataset


print("Collecte methode hybrid :")
hybrid_data = collect_results(test_cases, method="hybrid", limit=3)
print(f"\n{len(hybrid_data)} samples prets")

Collecte methode hybrid :

3 samples prets


In [13]:
print(json.dumps(hybrid_data, indent=2, ensure_ascii=False))

[
  {
    "user_input": "Inquiry About Secure Medical Data Solutions",
    "retrieved_contexts": [
      "Customer Support, I am reaching out to inquire about the medical data practices and hospital solutions used by your organization. As a healthcare professional, I understand the importance of protecting sensitive patient information. Could you provide information on the latest security protocols and measures that have been implemented to ensure the confidentiality and integrity of medical data? I would appreciate any recommendations for hospital solutions that prioritize data security. Additionally, could you let me know about the relevant regulations and compliance standards that must be met? Thank you.",
      "Could you provide more details on the solutions offered for securing medical data?",
      "I am reaching out to inquire about secure medical data solutions for hospital services, with a particular focus on billing integration. Could you provide details about the services t

## 5. Evaluation avec RAGAS (hybrid)

In [22]:
from ragas.metrics import ContextPrecision
from ragas import EvaluationDataset, evaluate

# 1. Init
context_precision = ContextPrecision(llm=evaluator_llm)
eval_dataset = EvaluationDataset.from_list(hybrid_data)

# 2. Eval
results = evaluate(
    dataset=eval_dataset,
    metrics=[context_precision]
)

# 3. Conversion en DataFrame pour plus de sécurité
df_results = results.to_pandas()

# 4. Calcul de la moyenne via Pandas pour éviter l'erreur de formatage
mean_val = df_results['context_precision'].mean()

print(f"--- Résultats de l'évaluation ---")
print(f"Score Moyen Context Precision : {mean_val:.4f}")

# 5. Affichage propre
import pandas as pd
display(df_results[['user_input', 'context_precision']])

C:\Users\mahmo\AppData\Local\Temp\ipykernel_11980\3800018537.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision
Evaluating: 100%|██████████| 3/3 [00:25<00:00,  8.59s/it]


--- Résultats de l'évaluation ---
Score Moyen Context Precision : 0.1944


,user_input,context_precision
0,Inquiry About Secure Medical Data Solutions,0.583333
1,Enhancing Multi-Unit Marketing Processes,0.000000
2,Enhance Investment Analysis Software,0.000000


## 6. Evaluation complete 

In [25]:
from ragas.metrics import ContextPrecision, ContextRecall, ContextEntityRecall
from ragas import EvaluationDataset, evaluate

# 3 metriques retrieval (pas de generation LLM dans notre pipeline)
all_metrics = [
    ContextPrecision(llm=evaluator_llm),
    ContextRecall(llm=evaluator_llm),
    ContextEntityRecall(llm=evaluator_llm),
]

eval_dataset = EvaluationDataset.from_list(hybrid_data)

results = evaluate(
    dataset=eval_dataset,
    metrics=all_metrics,
    raise_exceptions=False,
    show_progress=True,
    batch_size=1,
)

# Resultats en DataFrame
df_results = results.to_pandas()

# Colonnes de scores uniquement
score_cols = [c for c in df_results.columns if c not in ["user_input", "retrieved_contexts", "response", "reference"]]

# Moyennes par metrique
print("=== SCORES MOYENS (hybrid) ===")
for col in score_cols:
    mean_val = df_results[col].mean()
    print(f"  {col}: {mean_val:.4f}")

print()
display(df_results[["user_input"] + score_cols])

C:\Users\mahmo\AppData\Local\Temp\ipykernel_11980\2050944156.py:1: DeprecationWarning: Importing ContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextPrecision
  from ragas.metrics import ContextPrecision, ContextRecall, ContextEntityRecall
C:\Users\mahmo\AppData\Local\Temp\ipykernel_11980\2050944156.py:1: DeprecationWarning: Importing ContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ContextRecall
  from ragas.metrics import ContextPrecision, ContextRecall, ContextEntityRecall
C:\Users\mahmo\AppData\Local\Temp\ipykernel_11980\2050944156.py:1: DeprecationWarning: Importing ContextEntityRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collect

=== SCORES MOYENS (hybrid) ===
  context_precision: 0.1944
  context_recall: 0.2222
  context_entity_recall: 0.2056



,user_input,context_precision,context_recall,context_entity_recall
0,Inquiry About Secure Medical Data Solutions,0.583333,0.666667,0.200000
1,Enhancing Multi-Unit Marketing Processes,0.000000,0.000000,0.166667
2,Enhance Investment Analysis Software,0.000000,0.000000,0.250000
